# SpliceBERT Head–Context Atlas — Stage 2 A3 recovery

Run **Runtime → Run all**. This notebook never reruns the completed target forwards. It preserves the failed runtime manifest, applies the outcome-silent A3 numerical audit to the existing 4,019 shards, and withholds mechanism labels until the independent analysis audit passes.

In [ ]:
# 1. Restore the audited code overlay and verify the preserved failed run
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib, json, shutil, zipfile

base_zip = Path('/content/drive/MyDrive/gfm_atlas_screen_colab_v1.zip')
drive_root = Path('/content/drive/MyDrive/gfm_atlas_screen_v1_results')
overlay = drive_root / 'gfm_atlas_stage2_colab_overlay_v6.zip'
sidecar = drive_root / 'gfm_atlas_stage2_colab_overlay_v6.zip.sha256.json'
repo_root = Path('/content/gfm_atlas_screen')
repo = repo_root / 'repo'
stage2_out = Path('/content/drive/MyDrive/gfm_atlas_stage2_v1_results')

assert base_zip.is_file(), base_zip
assert overlay.is_file() and sidecar.is_file(), (overlay, sidecar)
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()
expected = json.loads(sidecar.read_text())['sha256']
assert sha256(overlay) == expected

# A clean local extraction changes no Drive result artifact.
if repo_root.exists():
    shutil.rmtree(repo_root)
with zipfile.ZipFile(base_zip) as archive:
    archive.extractall('/content')
with zipfile.ZipFile(overlay) as archive:
    archive.extractall(repo)
overlay_manifest = json.loads((repo / 'STAGE2_OVERLAY_MANIFEST.json').read_text())
assert overlay_manifest['schema'] == 'head_context_atlas.stage2_colab_overlay/6'

health_manifest = json.loads((stage2_out / 'run_manifest.health.json').read_text())
target_manifest_path = stage2_out / 'run_manifest.target.json'
target_manifest = json.loads(target_manifest_path.read_text())
assert health_manifest['status'] == 'PASS' and health_manifest['n_failures'] == 0
assert len(list((stage2_out / 'health_shards').glob('packet_*.json'))) == 256
assert len(list((stage2_out / 'target_shards').glob('packet_*.json'))) == 4019
assert target_manifest['status'] == 'FAIL' and target_manifest['n_failures'] == 1
assert [row['gate'] for row in target_manifest['failures']] == ['target_whole_stage_reference']
assert target_manifest['failures'][0]['threshold'] == 1e-6
assert target_manifest['failures'][0]['value'] <= 1e-4
print('A3_RECOVERY_PREFLIGHT_PASS | 4019 immutable target shards | labels still sealed')

In [ ]:
# 2. Independently audit, merge, analyze, and reconstruct under A3
import os, subprocess, sys
env = os.environ.copy()
env['PYTHONHASHSEED'] = '0'
env['TOKENIZERS_PARALLELISM'] = 'false'
package = 'results/splicebert_head_context_atlas/atlas_stage2_package_v1.json'
package_audit = 'results/splicebert_head_context_atlas/atlas_stage2_package_v1.audit.json'
execution_audit = stage2_out / 'validation_audit.a3_v2.json'
merged = stage2_out / 'atlas_stage2_mechanism_v1_merged_a3_v2'
analysis = stage2_out / 'atlas_stage2_mechanism_analysis_v1_a3_v2'

steps = [
    (execution_audit, [sys.executable, 'scripts/splicebert_head_context_atlas/validate_atlas_stage2_mechanisms.py', '--run-dir', str(stage2_out), '--package', package, '--package-audit', package_audit, '--audit', str(execution_audit), '--a3-numerical-amendment']),
    (Path(str(merged) + '.manifest.json'), [sys.executable, 'scripts/splicebert_head_context_atlas/merge_atlas_stage2_mechanisms.py', '--run-dir', str(stage2_out), '--audit', str(execution_audit), '--package', package, '--prefix', str(merged)]),
    (Path(str(analysis) + '.summary.json'), [sys.executable, 'scripts/splicebert_head_context_atlas/analyze_atlas_stage2_mechanisms.py', '--merged-prefix', str(merged), '--package', package, '--package-audit', package_audit, '--execution-audit', str(execution_audit), '--run-dir', str(stage2_out), '--prefix', str(analysis)]),
    (Path(str(analysis) + '.audit.json'), [sys.executable, 'scripts/splicebert_head_context_atlas/validate_atlas_stage2_analysis.py', '--merged-prefix', str(merged), '--analysis-prefix', str(analysis), '--package', package, '--run-dir', str(stage2_out)]),
]
for sentinel, command in steps:
    if sentinel.is_file():
        record = json.loads(sentinel.read_text())
        assert record.get('status') in {'PASS', 'STAGE2_CLASSIFIED', 'STAGE2_MERGED'}, (sentinel, record.get('status'))
        print('Existing immutable step:', sentinel.name)
    else:
        subprocess.run(command, cwd=repo, env=env, check=True)
final_audit = json.loads(Path(str(analysis) + '.audit.json').read_text())
assert final_audit['status'] == 'PASS' and final_audit['n_failures'] == 0, final_audit
print('STAGE2_A3_ANALYSIS_AUDIT_PASS | failures: 0')

In [ ]:
# 3. Only now unseal the audited conditional-mechanism results
final_audit = json.loads(Path(str(analysis) + '.audit.json').read_text())
assert final_audit['status'] == 'PASS' and final_audit['n_failures'] == 0
a3 = json.loads(execution_audit.read_text())
relations = json.loads(Path(str(analysis) + '.relations.json').read_text())
print('A3 numerical audit:', json.dumps(a3['phase_reports']['target']['whole_reference_numerical_audit'], indent=2))
print('Audited relations:', json.dumps(relations, indent=2))

In [ ]:
# 4. Build and independently audit the eight representative evidence bundles
dag = stage2_out / 'atlas_representative_dags_v1_a3_v2.json'
dag_manifest = dag.with_suffix('.manifest.json')
dag_audit = dag.with_suffix('.audit.json')
if not dag.is_file():
    subprocess.run([sys.executable, 'scripts/splicebert_head_context_atlas/build_atlas_representative_dags.py', '--package', package, '--package-audit', package_audit, '--analysis-prefix', str(analysis), '--analysis-audit', str(Path(str(analysis) + '.audit.json')), '--stage2-run-dir', str(stage2_out), '--cohort-prefix', 'results/splicebert_head_context_atlas/atlas_context_matched_v7', '--output', str(dag), '--device', 'cuda'], cwd=repo, env=env, check=True)
assert dag.is_file() and dag_manifest.is_file()
if not dag_audit.is_file():
    subprocess.run([sys.executable, 'scripts/splicebert_head_context_atlas/validate_atlas_representative_dags.py', '--bundle', str(dag), '--package', package, '--package-audit', package_audit, '--analysis-prefix', str(analysis), '--analysis-audit', str(Path(str(analysis) + '.audit.json')), '--audit', str(dag_audit)], cwd=repo, env=env, check=True)
dag_report = json.loads(dag_audit.read_text())
assert dag_report['status'] == 'PASS' and dag_report['n_failures'] == 0 and dag_report['export_eligible'] is True
print('REPRESENTATIVE_DAG_AUDIT_PASS | representatives:', dag_report['n_representatives'])